## Creating the ESG Subscription Agent

### Installing Utilities and Libraries

In [ ]:
%pip install azure-servicebus==7.14.3 openai==2.38.0 python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# loading service bus configurations
service_bus_connection_string = os.getenv("SERVICE_BUS_CONNECTION_STRING")
service_bus_topic_name = os.getenv("SERVICE_BUS_TOPIC_NAME")
service_bus_esg_agent_subscription_name = os.getenv("SERVICE_BUS_ESG_AGENT_SUBSCRIPTION_NAME")
service_bus_microsoft_agent_subscription_name = os.getenv("SERVICE_BUS_MICROSOFT_AGENT_SUBSCRIPTION_NAME")

# loading the azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model = os.getenv("CHAT_COMPLETIONS_MODEL")

### Creating the Service Bus Client

In [ ]:
from azure.servicebus import ServiceBusClient

sb_client = ServiceBusClient.from_connection_string(
    conn_str = service_bus_connection_string
)

### Helper Function to Process User Queries

In [ ]:
from openai import AzureOpenAI

def process_user_message(user_query, llm):
    azure_openai_client = AzureOpenAI(
        azure_endpoint = azure_openai_endpoint,
        api_version = "2024-06-01",
        api_key = azure_openai_api_key
    )

    response = azure_openai_client.chat.completions.create(
        model = llm,
        messages=[
            {
                "role": "system",
                "content": """You are a helpful ESG agent. Answer user queries on ESG related topics and
                              frameworks like ESRS, BRSR, TCFD, GRI and UN SDGs"""
            },
            {
                "role": "user",
                "content": user_query
            }
        ],
        temperature = 0.7
    )

    return response.choices[0].message.content

### Process Messages Reliably from the Subscription

In [ ]:
import json
from azure.servicebus import ServiceBusReceiveMode

# creating the subscription consumer object
consumer = sb_client.get_subscription_receiver(topic_name = service_bus_topic_name,
                                               subscription_name = service_bus_esg_agent_subscription_name,
                                               receive_mode = ServiceBusReceiveMode.PEEK_LOCK,
                                               max_wait_time = 60)

for msg in consumer:
    body = json.loads(str(msg))
    
    # process message
    print("user query: {} \n".format(body.get("prompt")))
    assistant_response = process_user_message(body.get("prompt"), body.get("model"))
    print("Assistant response: {}".format(assistant_response))
    print("=================================================")
    print("\n")

    # mark the message as complete after successful execution
    consumer.complete_message(msg)